# Inspect a live Scenario

Load a scenario script and walk every inspection surface before running the simulation.

**Before you click "Run All":** `load_scenario_from_path` imports the scenario
module, which calls `load_or_build_world` internally. On a cache miss,
`load_or_build_world` shows an interactive consent prompt and raises
`LLMBuildAbortedError` in non-TTY contexts (CI, scheduled jobs). The demo
below uses `scenarios/example_llm_world_offline.py` (6-item fashion catalog,
no OpenAI key required), which always builds its world from a `CannedClient`
and never touches the network. Be careful if you swap it for a scenario that
calls `WorldBuilder` against a live LLM client — you may trigger an LLM build
on first use.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append('../')

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

from src.sim.scenario import load_scenario_from_path


## Load a scenario

Point `SCENARIO_PATH` at any scenario script that exposes a top-level
`scenario` attribute. The offline script builds its `World` from a
`CannedClient`, so it is deterministic and requires no OpenAI key.


In [ ]:
SCENARIO_PATH = '../scenarios/example_llm_world_offline.py'

scenario = load_scenario_from_path(SCENARIO_PATH)
print(f'Loaded scenario: {len(scenario.catalog)} catalog items, '
      f'{len(scenario.stores)} stores, '
      f'{scenario.n_steps} steps')


### Summary — top-level simulation parameters

One-row overview: step count, start date, world seed, and the size of the
catalog and store roster. Use this as a quick sanity check that you loaded
the right scenario before inspecting the detail views below.


In [ ]:
scenario.summary_df()


### Catalog — items for sale

One row per SKU. `margin = base_price - unit_cost` is derived. `freshness_alpha`
and `freshness_decay` govern the per-SKU hype-curve multiplier; `None` means
the simulation falls back to the `ItemLifecycleParams` defaults. `stage_change_probs`
overrides the global transition table for that SKU when set.


In [ ]:
scenario.catalog_df()


### Stores — retail roster with live policies

One row per `StoreInstance`. `policy_class` is populated from the live
`Policy` object attached by the scenario script — this is the key difference
from a historical-run artifact loaded via `Scenario.from_json`, where
`policy_class` is always `None` (policies are not serialised). Non-`None`
values here confirm that live policies survived the `load_scenario_from_path`
call.


In [ ]:
scenario.stores_df()


### Market — demand and seasonality parameters

Domain-meaningful fields (`cycle_len`, `peak_factor`, `off_factor`,
`init_demand`, `price_elasticity`, `regions`, `season_months`) are LLM-authored;
math defaults (`sigma`, `crn_draws`, clamp bounds, etc.) are merged in by
`WorldBuilder`. One-row DataFrame — transpose with `.T` for easier reading
if the column list is long.


In [ ]:
scenario.market_df().T


### Disruption — supply-chain shock parameters

Governs random disruption events: `event_prob` is the per-step probability,
`types` lists the event categories, `severity` and `duration` are
`Distribution` objects sampled at event time. The `regions` list scopes
which stores are exposed.


In [ ]:
scenario.disruption_df()


### Lifecycle — product stage defaults

Catalog-wide defaults for the two-layer lifecycle clock (ADR 0001). `stages`
is the canonical ordered list; `init_stage` seeds every SKU at step 0 unless
the `Ware` carries a per-item override. `default_stage_change_probs` is the
transition table; `default_freshness_alpha` / `default_freshness_decay` are
the fallback hype-curve parameters used when `Ware.freshness_alpha` is `None`.


In [ ]:
scenario.lifecycle_df()


## (Optional) Historical-run audit via `Scenario.from_json`

After running a simulation, the runner serialises the scenario to
`data/<run>/config/scenario.json`. Load it back with `Scenario.from_json` to
inspect the exact catalog, market, and disruption parameters used in a past
run.

**Note:** `policy_class` is always `None` in the `stores_df()` of a
deserialized scenario — this is by design, not a bug. Policies are wired up
in the experiment script and are never serialised (the LLM never authors them).
Re-attach policies with `make_stores` if you need to re-run the same scenario.

The cell below expects `data/example_llm_world_offline/config/scenario.json`
to exist. Run `uv run python main.py scenarios/example_llm_world_offline.py`
once to generate it.


In [ ]:
from src.sim.scenario import Scenario

SCENARIO_JSON_PATH = Path('../data/example_llm_world_offline/config/scenario.json')

if SCENARIO_JSON_PATH.exists():
    scenario_from_run = Scenario.from_json(SCENARIO_JSON_PATH.read_text())
    print('Loaded from historical run artifact')
    display(scenario_from_run.summary_df())
    print('\nstores_df() — note policy_class is None (by design):')
    display(scenario_from_run.stores_df())
else:
    print(
        f'Artifact not found: {SCENARIO_JSON_PATH}\n'
        'Run the offline scenario first:\n'
        '  uv run python main.py scenarios/example_llm_world_offline.py'
    )
